# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/flyrank-ml-internship-abdurrehman/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook synthesizes weeks 1–7 (`w01`–`w07` in this folder) into the shape of the deployed paper. It does not re-derive new numbers — it pulls the already-validated results from those notebooks, in particular the honest, leakage-checked model from `w06_validation_audit.ipynb` and the ranked queue from `w07_action_playbook.ipynb`.

> Deployed paper: see `submission/paper_url.txt` for the live URL, or `docs/index.html` in this repo.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
# QUESTION: Which specific, currently declining pages should the content team
# review first? (Lane 2 — Refresh / Content Opportunity Scoring, from w01_research_question.ipynb)
#
# Decision supported: a ranked, human-reviewed content-refresh queue.
# Action: a reviewer opens the top-ranked page and decides refresh / consolidate / leave.
# Cost of a wrong call: a reviewer wastes hours on a low-value, seasonal, or dead page
# while a higher-value opportunity sits unreviewed further down the list.
#
# Early evidence this lane was worth pursuing (from a 30,000-page starter slice,
# w01_research_question.ipynb): 16,262 pages showed a downward trend, and 9,956 of
# those had >500 impressions in the last 90 days — a substantial, high-value backlog.
print("Lane: Refresh / Content Opportunity Scoring")
print("Decision: which declining pages should a human reviewer open first?")


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
# DATA: FlyRank ML Internship warehouse (fact_content_daily_performance) plus a
# trailing-90-day anonymized content-performance slice used for modeling, spanning
# 32 clients and 3 content types. A separate month-slice check (March 2026) confirmed
# 9,841,378 rows in a single month, consistent with the program's ~79M-row scale
# (w03_data_contract.ipynb).
#
# Grain verified: one row per (report_date, client, content) — max rows per group
# checked at 1 (expected).
#
# EXCLUDED and why:
# - client_id / content_id: grouping only, never model features (privacy + leakage risk)
# - trend_direction, trend_pct: is_declining_label is derived from these — including
#   them would hand the model the answer (confirmed via the leakage-trap experiment
#   in w03: accuracy WITH a leakage trap hit 1.0000 vs 0.5256 without)
# - No client names, raw URLs, or verbatim queries appear anywhere in this project.
#
# DATA LIMIT: no report_date-style field exists in the modeling CSV, only a duration
# (days_since_last_update) — so every split in this project is grouped-by-client,
# not a genuine time-aware split. This is called out explicitly in Limitations.
print("Data grain: one row per (report_date, client, content); verified max group size = 1")
print("Scale check (single month): 9,841,378 rows")


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
# LABEL: is_declining_label = 1 if trend_pct < 0, else 0.
#
# BASELINE (w04_baseline_score.ipynb): stale_and_slipping rule — flag if
# days_since_last_update >= 180 AND avg_position > 10 AND position data exists.
# On the held-out test split: Precision 0.611, Recall 0.463, Accuracy 0.478.
#
# MODEL: Logistic Regression, chosen for interpretability — a readable linear model
# makes leakage immediately visible via its coefficients (w05_model.ipynb).
#
# VALIDATION DESIGN: GroupShuffleSplit(test_size=0.2), grouped by client_id, so no
# client appears in both train and test. A time-aware split was considered and
# rejected — the only duration field available is not a calendar date, so splitting
# on it would silently split by freshness tier, not by past-vs-future.
#
# LEAKAGE CHECK — THE KEY FINDING: the first model (w05) scored Precision 1.00,
# Recall 0.999, Accuracy 1.00 — a red flag, not a success. A confession test
# (w06_validation_audit.ipynb) removed the two top-coefficient features
# (impressions_last_30d: -7.92, impressions_prev_30d: +7.91 — near-cancelling,
# a classic leakage signature) and accuracy collapsed to ~0.672, in line with
# the base rate (0.628) plus a modest real signal. A second issue — avg_position
# == 0 being read as a literal top rank instead of "no data" — was also found
# and fixed via an explicit has_avg_position flag.
with_leak = {"precision": 1.000, "recall": 1.000, "accuracy": 1.000}
without_leak = {"precision": 0.714, "recall": 0.798, "accuracy": 0.672, "base_rate": 0.628}
print("WITH leakage (impressions_*_30d included):", with_leak)
print("WITHOUT leakage (honest model):", without_leak)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# RESULTS on the held-out test split (n=6,163), grouped by client_id:
#
# | Metric    | Rule baseline | Honest model |
# |-----------|--------------:|-------------:|
# | Precision |         0.611 |        0.714 |
# | Recall    |         0.463 |        0.798 |
# | Accuracy  |         0.478 |        0.672 |
# | Base rate |              0.628 (both)    |
#
# For a RANKED queue, precision@K matters more than global accuracy
# (w07_action_playbook.ipynb): precision@100 = 0.880, precision@250 = 0.860 —
# both well above global precision (0.714) and the base rate (0.628).
#
# CAVEAT: the grouped holdout used for precision@K happened to land on 7 clients
# whose content is 100% "keyword article" — precision@K is directly demonstrated
# for that content type, not independently confirmed for the other two the model
# was trained on. Flagged explicitly in Limitations.
import pandas as pd
results = pd.DataFrame({
    "Metric": ["Precision", "Recall", "Accuracy"],
    "Rule Baseline": [0.611, 0.463, 0.478],
    "Honest Model": [0.714, 0.798, 0.672],
})
print(results.to_string(index=False))
print("\nprecision@100 = 0.880 | precision@250 = 0.860 | base rate = 0.628")


## 5. Limitations

*What this work cannot claim.*

In [ ]:
# - Global precision (0.714) is modest — meaningfully above the 62.8% base rate
#   but far from a strong standalone predictor; value is concentrated in ranking.
# - No genuine time-based validation is possible with this dataset (no calendar
#   date field) — cannot claim "this page will decline next month."
# - Single train/test holdout, not cross-validated — exact precision@K would
#   likely shift under a different seed.
# - Precision@K validated directly for keyword articles only, out of 3 trained
#   content types (see caveat in Section 4).
# - Separate program research claims audited during this internship (e.g. large
#   reported lifts from refreshing aged pages) were found to rest on non-random
#   editor selection of which pages got refreshed — a selection-bias issue, not
#   leakage — meaning those reported lifts mix the refresh effect with the
#   choosing effect (full audit in w06_validation_audit.ipynb, Section 1).
# - This is decision support, not causal proof: nothing here shows that acting
#   on the queue causes recovery, only that the ranking beats a rule baseline
#   and an unranked list on this historical slice.
print("See markdown cell above for the full limitations list.")


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# From w07_action_playbook.ipynb:
# 1. Use the queue top-down — precision is highest in the top ~100-250 ranked items.
# 2. Route "no ranking data" items (avg_position was unmeasured, ~1% of the queue)
#    to a data-verification step, never treat as "ranked last."
# 3. Re-validate with a stratified split before extending the queue to feedly or
#    comparison articles (see Section 4 caveat).
# 4. Never auto-publish or auto-edit from this queue — human review only.
# 5. Monitor for the leakage signature reappearing in any newly added feature.
# 6. Track base-rate drift (>5pt move from 62.8%) as a retrain trigger.
recommendations = [
    "Use the queue top-down, not as a flat list",
    "Route missing-data rows to verification, not the action queue",
    "Re-validate before extending to other content types",
    "Never auto-publish or auto-edit from model output",
    "Monitor for the leakage signature reappearing",
    "Track base-rate drift as a retrain trigger",
]
for i, r in enumerate(recommendations, 1):
    print(f"{i}. {r}")


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
# The three charts embedded in the deployed paper (docs/index.html) are:
# 1. Model vs. baseline, same split (precision/recall/accuracy vs base rate)
# 2. Precision@K — showing the queue gets cleaner near the top (0.714 global ->
#    0.86 @250 -> 0.88 @100)
# 3. The leakage confession — accuracy collapsing from 1.00 to 0.672 once the
#    label-sibling features are removed
#
# These are generated directly from the numbers validated in w06 and w07 above
# and embedded as inline images in the deployed page for a self-contained,
# no-broken-image-path artifact. The ranked queue CSV itself
# (work/outputs/refresh_action_queue.csv) is produced by running
# w07_action_playbook.ipynb end to end, per work/README.md — it is not committed
# to the repo since work/outputs is gitignored by design (regenerate, don't commit).
print("Artifacts: 3 charts embedded in docs/index.html; ranked queue reproducible via w07.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.